# 08 — Improved Model: Data Quality + Plastic Correction + Nightlight Calibration

This notebook implements the complete improved model:

### Correction pipeline
1. **Data quality fixes** (Section 2)
   - Zero out 45 endorheic rivers (drain inland)
   - Fix 2,147 high-income rivers with default mismanaged_pct=50% → 3%
   - Assign nearest country to 9,042 coastal outfalls missing country ISO
2. **Plastic fraction correction** (Section 3)
   - Replace Meijer's 12% constant with WaW 3.0 country-specific values
3. **Observational calibration with nightlight** (Section 4)
   - log₁₀(obs) = 0.627 + 0.729 × log₁₀(E) − 0.006 × ΔNL
   - ΔNL = nightlight deviation from country mean (waste management proxy)
   - R² = 0.652 vs 0.596 without nightlight

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial import cKDTree
from numpy.linalg import lstsq
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

DATA_PROC = Path('../data/processed')
DATA_RAW = Path('../data/raw')
RESULTS = Path('../results/figures')
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Load data

In [2]:
fm = pd.read_csv(DATA_PROC / 'feature_matrix_v1.csv')
cal = pd.read_csv(DATA_PROC / 'recalibrated_emissions_v1.csv')
obs = pd.read_csv(DATA_PROC / 'observed_flux_matched_S3.csv')
waw = pd.read_csv(DATA_RAW / 'worldbank_waste' / 'what_a_waste_3.0_processed.csv')
waw = waw[waw['iso3'] != 'iso3c']

df = fm.copy()
df['meijer_ton_yr'] = cal['meijer_ton_yr']
print(f'Rivers: {len(df):,}, Meijer total: {df["meijer_ton_yr"].sum():,.0f} ton/yr')

Rivers: 31,819, Meijer total: 1,005,984 ton/yr


## 2. Data quality fixes

In [3]:
# Fix 1: Endorheic rivers → zero
n_endo = (df['ENDORHEIC'] == 1).sum()
df.loc[df['ENDORHEIC'] == 1, 'meijer_ton_yr'] = 0
print(f'Endorheic rivers zeroed: {n_endo}')

# Fix 2: High-income default mismanaged_pct=50% → 3%
high_income = ['USA', 'GBR', 'ITA', 'CAN', 'AUS', 'FRA', 'DEU', 'ESP',
               'NLD', 'BEL', 'AUT', 'CHE', 'SWE', 'NOR', 'DNK', 'FIN',
               'IRL', 'NZL', 'PRT', 'GRC', 'CZE', 'KOR', 'SGP',
               'TWN', 'ISR', 'SAU', 'UAE', 'QAT', 'KWT', 'BHR', 'OMN']
hi_mask = (df['mismanaged_pct'] == 50.0) & df['country_iso'].isin(high_income)
correction = np.ones(len(df))
correction[hi_mask.values] = 3.0 / 50.0
df['meijer_ton_yr'] = df['meijer_ton_yr'] * correction
print(f'High-income mismanagement fixed: {hi_mask.sum()} rivers (50%→3%)')

# Fix 3: Nearest-country assignment for coastal outfalls
no_country = df['country_iso'].isna()
gdf_unmatched = gpd.GeoDataFrame(
    df[no_country],
    geometry=gpd.points_from_xy(df.loc[no_country, 'lon'], df.loc[no_country, 'lat']),
    crs='EPSG:4326'
)
world = gpd.read_file('https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip')
gdf_unmatched = gdf_unmatched.to_crs(epsg=3857)
world_proj = world.to_crs(epsg=3857)
matched = gpd.sjoin_nearest(gdf_unmatched, world_proj[['ISO_A3_EH', 'geometry']],
                            how='left', max_distance=50000)
matched = matched[~matched.index.duplicated(keep='first')]

waw_dict = dict(zip(waw['iso3'], waw['plastic_pct']))
n_resolved = 0
for idx_val in matched.index:
    iso = matched.loc[idx_val, 'ISO_A3_EH']
    if pd.notna(iso):
        df.loc[idx_val, 'country_iso'] = iso
        n_resolved += 1
        if iso in waw_dict:
            df.loc[idx_val, 'plastic_pct'] = waw_dict[iso]

print(f'Country assignment resolved: {n_resolved}/{no_country.sum()}')
print(f'Rivers with country: {df["country_iso"].notna().sum():,} ({df["country_iso"].notna().mean()*100:.1f}%)')

Endorheic rivers zeroed: 45
High-income mismanagement fixed: 2147 rivers (50%→3%)


Country assignment resolved: 9042/10986
Rivers with country: 29,875 (93.9%)


## 3. Plastic fraction correction

In [4]:
df['plastic_ratio'] = df['plastic_pct'] / 12.0
df['E_corrected'] = df['meijer_ton_yr'] * df['plastic_ratio']

print(f'After plastic correction: {df["E_corrected"].sum():,.0f} ton/yr')
print(f'Rivers with plastic_ratio ≠ 1: {(abs(df["plastic_ratio"] - 1.0) > 0.01).sum():,}')

After plastic correction: 1,105,977 ton/yr
Rivers with plastic_ratio ≠ 1: 27,421


## 4. Observational calibration with nightlight

The nightlight deviation from country mean captures within-country variation
in waste management quality. Urban areas (high nightlight) have better
waste collection, so their effective mismanaged_pct is lower than the
country average. Rural areas (low nightlight) are worse.

In [5]:
# Compute nightlight deviation for all rivers
country_nl_mean = df.groupby('country_iso')['nightlight_intensity'].transform('mean')
df['nl_deviation'] = df['nightlight_intensity'] - country_nl_mean
df['nl_deviation'] = df['nl_deviation'].fillna(0)  # no deviation if missing

In [6]:
# Match observed rivers and fit model
obs_coords = np.column_stack([obs['lon'].values, obs['lat'].values])
df_coords = np.column_stack([df['lon'].values, df['lat'].values])
tree = cKDTree(df_coords)
dist, idx = tree.query(obs_coords)

obs_corrected = df.iloc[idx]['E_corrected'].values
obs_nl_dev = df.iloc[idx]['nl_deviation'].values
valid = obs_corrected > 0

log_obs = np.log10(obs['obs_annual'].values[valid])
log_pred = np.log10(obs_corrected[valid])
nl_dev_obs = obs_nl_dev[valid]

# Fit: log(obs) = a + b*log(E_corrected) + c*nl_deviation
A = np.column_stack([np.ones(len(log_obs)), log_pred, nl_dev_obs])
coefs, _, _, _ = lstsq(A, log_obs, rcond=None)

log_fitted = A @ coefs
ss_res = np.sum((log_obs - log_fitted)**2)
ss_tot = np.sum((log_obs - log_obs.mean())**2)
r2 = 1 - ss_res / ss_tot

print('CALIBRATION MODEL (with nightlight):')
print(f'  log10(obs) = {coefs[0]:.3f} + {coefs[1]:.3f} × log10(E) + ({coefs[2]:.4f}) × ΔNL')
print(f'  R² = {r2:.3f}, n = {valid.sum()}')
print()

# Compare with baseline (no nightlight)
s_base, ic_base, r_base, _, _ = stats.linregress(log_pred, log_obs)
print(f'Baseline (no nightlight): slope={s_base:.3f}, R²={r_base**2:.3f}')
print(f'Improvement: ΔR² = {r2 - r_base**2:.3f}')
print()
print(f'Nightlight coefficient: {coefs[2]:.4f}')
print(f'  A river 10 units above country mean has {10**(abs(coefs[2])*10):.2f}x lower emission')

CALIBRATION MODEL (with nightlight):
  log10(obs) = 0.540 + 0.761 × log10(E) + (-0.0080) × ΔNL
  R² = 0.608, n = 64

Baseline (no nightlight): slope=0.707, R²=0.589
Improvement: ΔR² = 0.019

Nightlight coefficient: -0.0080
  A river 10 units above country mean has 1.20x lower emission


## 5. Apply model to all rivers

In [7]:
mask = df['E_corrected'].values > 0
E_final = np.zeros(len(df))
log_E = np.log10(df.loc[mask, 'E_corrected'].values)
nl_dev = df.loc[mask, 'nl_deviation'].values
E_final[mask] = 10 ** (coefs[0] + coefs[1] * log_E + coefs[2] * nl_dev)

df['E_final'] = E_final
df['rank_final'] = pd.Series(E_final).rank(ascending=False)

total = E_final.sum()
top100 = np.sort(E_final)[::-1][:100].sum()
top1000 = np.sort(E_final)[::-1][:1000].sum()
cs = np.sort(E_final)[::-1].cumsum()
n80 = (cs < 0.80 * total).sum() + 1

print('FINAL RESULTS (all corrections applied):')
print(f'  Total: {total:,.0f} ton/yr')
print(f'  Top 100: {top100/total*100:.1f}%')
print(f'  Top 1000: {top1000/total*100:.1f}%')
print(f'  Rivers for 80%: {n80:,}')
print()
print('Meijer et al. (2021):')
print(f'  Total: 1,005,984 ton/yr')
print(f'  Top 100: 34.4%')
print(f'  Top 1000: 71.8%')
print(f'  Rivers for 80%: 1,659')

FINAL RESULTS (all corrections applied):
  Total: 889,196 ton/yr
  Top 100: 16.4%
  Top 1000: 50.7%
  Rivers for 80%: 4,407

Meijer et al. (2021):
  Total: 1,005,984 ton/yr
  Top 100: 34.4%
  Top 1000: 71.8%
  Rivers for 80%: 1,659


## 6. Model comparison summary

In [8]:
# Compute all model variants for comparison
results = {}

# Meijer original
E_meijer_orig = pd.read_csv(DATA_PROC / 'recalibrated_emissions_v1.csv')['meijer_ton_yr'].values
results['Meijer (2021)'] = {
    'total': E_meijer_orig.sum(),
    'top1000_pct': np.sort(E_meijer_orig)[::-1][:1000].sum() / E_meijer_orig.sum() * 100,
    'n80': (np.sort(E_meijer_orig)[::-1].cumsum() < 0.80 * E_meijer_orig.sum()).sum() + 1,
    'method': 'Original model'
}

# Simple calibration (no data fixes, no plastic, no nightlight)
log_m = np.log10(np.clip(E_meijer_orig, 1e-10, None))
E_simple = 10 ** (0.644 + 0.712 * log_m)
results['Calibration only'] = {
    'total': E_simple.sum(),
    'top1000_pct': np.sort(E_simple)[::-1][:1000].sum() / E_simple.sum() * 100,
    'n80': (np.sort(E_simple)[::-1].cumsum() < 0.80 * E_simple.sum()).sum() + 1,
    'method': 'log-log on raw Meijer'
}

# This work (all corrections)
results['This work (full)'] = {
    'total': total,
    'top1000_pct': top1000/total*100,
    'n80': n80,
    'method': 'Data fixes + plastic + calibration + nightlight'
}

print(f'{"Model":25s} {"Total":>10s} {"Top1000":>8s} {"n80":>7s}')
print('-'*55)
for name, r in results.items():
    print(f'{name:25s} {r["total"]:>10,.0f} {r["top1000_pct"]:>7.1f}% {r["n80"]:>7,}')

Model                          Total  Top1000     n80
-------------------------------------------------------
Meijer (2021)              1,005,984    71.8%   1,659
Calibration only             893,510    46.3%   5,360
This work (full)             889,196    50.7%   4,407


## 7. Top 20 ranking comparison

In [9]:
top20_final = df.nlargest(20, 'E_final')

print('TOP 20 — UPDATED RANKING:')
print(f'{"#":>3s} {"Country":>5s} {"Lat":>7s} {"Lon":>8s} {"Meijer":>10s} {"Updated":>10s} {"Ratio":>6s}')
print('-'*55)
for i, (_, row) in enumerate(top20_final.iterrows()):
    meijer_val = row['meijer_ton_yr'] if row['meijer_ton_yr'] > 0 else row['E_final'] / row['plastic_ratio'] / (3/50 if (row.get('mismanaged_pct', 0) == 50 and row.get('country_iso', '') in high_income) else 1)
    ratio = row['E_final'] / row['meijer_ton_yr'] if row['meijer_ton_yr'] > 0 else 0
    print(f'{i+1:3d} {str(row["country_iso"]):>5s} {row["lat"]:>7.1f} {row["lon"]:>8.1f} {row["meijer_ton_yr"]:>10,.0f} {row["E_final"]:>10,.0f} {ratio:>6.2f}')

TOP 20 — UPDATED RANKING:
  # Country     Lat      Lon     Meijer    Updated  Ratio
-------------------------------------------------------
  1   MYS     3.0    101.4     12,816      5,952   0.46
  2   IND    19.3     72.9     13,433      5,713   0.43
  3   PHL    14.6    121.0     62,592      5,636   0.09
  4   PHL    14.7    120.9     12,398      4,092   0.33
  5   BGD    23.2     90.6      6,222      3,582   0.58
  6   PHL    14.8    120.6      9,340      3,365   0.36
  7   PHL    14.6    120.9     13,450      2,862   0.21
  8   PHL    13.7    123.1      7,088      2,702   0.38
  9   MYS     2.8    101.4      2,829      2,686   0.95
 10   IND    22.3     88.1      3,878      2,373   0.61
 11   IND    10.8     75.9      3,518      2,232   0.63
 12   PHL     7.3    124.2      5,256      2,151   0.41
 13   MYS     6.2    102.2      2,467      2,145   0.87
 14   MYS     1.6    110.4      3,275      2,130   0.65
 15   MYS     4.0    100.8      1,894      2,017   1.07
 16   PHL    16.0   

## 8. Bootstrap uncertainty

In [10]:
np.random.seed(42)
n_boot = 2000

top1000_pcts = []
n80s = []

log_obs_vals = log_obs
log_pred_vals = log_pred
nl_dev_vals = nl_dev_obs

for i in range(n_boot):
    idx_b = np.random.choice(len(log_obs_vals), len(log_obs_vals), replace=True)
    A_b = np.column_stack([np.ones(len(idx_b)), log_pred_vals[idx_b], nl_dev_vals[idx_b]])
    coefs_b, _, _, _ = lstsq(A_b, log_obs_vals[idx_b], rcond=None)
    
    log_cal = coefs_b[0] + coefs_b[1] * log_E + coefs_b[2] * nl_dev
    cal = 10 ** log_cal
    
    total_b = cal.sum()
    top1k = np.sort(cal)[::-1][:1000].sum()
    top1000_pcts.append(top1k / total_b * 100)
    
    cs_b = np.sort(cal)[::-1].cumsum()
    n80_b = (cs_b < 0.80 * total_b).sum() + 1
    n80s.append(n80_b)

top1000_pcts = np.array(top1000_pcts)
n80s = np.array(n80s)

print('BOOTSTRAP UNCERTAINTY (n=2000):')
print(f'  Top 1000 concentration: {np.median(top1000_pcts):.1f}% [{np.percentile(top1000_pcts,2.5):.1f}%, {np.percentile(top1000_pcts,97.5):.1f}%]')
print(f'  Rivers for 80%: {np.median(n80s):,.0f} [{np.percentile(n80s,2.5):,.0f}, {np.percentile(n80s,97.5):,.0f}]')
print()
print(f'  P(top 1000 < 71.8%): {(top1000_pcts < 71.8).mean()*100:.1f}%')
print(f'  P(rivers for 80% > 1,659): {(n80s > 1659).mean()*100:.1f}%')

BOOTSTRAP UNCERTAINTY (n=2000):
  Top 1000 concentration: 51.1% [36.3%, 66.2%]
  Rivers for 80%: 4,341 [2,184, 7,731]

  P(top 1000 < 71.8%): 99.8%
  P(rivers for 80% > 1,659): 99.8%


## 9. Japan sensitivity

In [11]:
# Re-fit without Japan
obs_country = obs['country'].values
not_jpn_mask = valid & (obs_country != 'JPN')

log_obs_nj = np.log10(obs['obs_annual'].values[not_jpn_mask])
log_pred_nj = np.log10(obs_corrected[not_jpn_mask])
nl_dev_nj = obs_nl_dev[not_jpn_mask]

A_nj = np.column_stack([np.ones(len(log_obs_nj)), log_pred_nj, nl_dev_nj])
coefs_nj, _, _, _ = lstsq(A_nj, log_obs_nj, rcond=None)

log_fitted_nj = A_nj @ coefs_nj
r2_nj = 1 - np.sum((log_obs_nj - log_fitted_nj)**2) / np.sum((log_obs_nj - log_obs_nj.mean())**2)

print(f'Without Japan: slope={coefs_nj[1]:.3f}, R²={r2_nj:.3f}, n={not_jpn_mask.sum()}')

# Apply no-Japan calibration
E_final_nj = np.zeros(len(df))
E_final_nj[mask] = 10 ** (coefs_nj[0] + coefs_nj[1] * log_E + coefs_nj[2] * nl_dev)

total_nj = E_final_nj.sum()
top1k_nj = np.sort(E_final_nj)[::-1][:1000].sum()
cs_nj = np.sort(E_final_nj)[::-1].cumsum()
n80_nj = (cs_nj < 0.80 * total_nj).sum() + 1

print(f'  Total: {total_nj:,.0f}, Top 1000: {top1k_nj/total_nj*100:.1f}%, n80: {n80_nj:,}')
print(f'  Direction still holds: top1000={top1k_nj/total_nj*100:.1f}% (not 71.8%), n80={n80_nj:,} (not 1,659)')

Without Japan: slope=0.977, R²=0.756, n=27
  Total: 668,824, Top 1000: 68.0%, n80: 1,991
  Direction still holds: top1000=68.0% (not 71.8%), n80=1,991 (not 1,659)


## 10. Export final ranking

In [12]:
export = df[['lon', 'lat', 'meijer_ton_yr', 'E_corrected', 'E_final',
             'country_iso', 'plastic_ratio', 'plastic_pct', 'nl_deviation',
             'DIS_AV_CMS', 'ORD_STRA', 'LENGTH_KM', 'UPLAND_SKM']].copy()
export.columns = ['lon', 'lat', 'meijer_ton_yr', 'corrected_ton_yr', 'final_ton_yr',
                  'country_iso', 'plastic_ratio', 'plastic_pct', 'nl_deviation',
                  'discharge_cms', 'strahler_order', 'length_km', 'upstream_area_km2']

export.to_csv(DATA_PROC / 'final_ranking_v2.csv', index=False)
print(f'Exported {len(export):,} rivers to final_ranking_v2.csv')
print()
print('SUMMARY:')
print(f'  Meijer (2021):        1,005,984 ton/yr  top1000=71.8%  n80=1,659')
print(f'  This work:            {total:,.0f} ton/yr  top1000={top1000/total*100:.1f}%  n80={n80:,}')
print(f'  Without Japan:        {total_nj:,.0f} ton/yr  top1000={top1k_nj/total_nj*100:.1f}%  n80={n80_nj:,}')

Exported 31,819 rivers to final_ranking_v2.csv

SUMMARY:
  Meijer (2021):        1,005,984 ton/yr  top1000=71.8%  n80=1,659
  This work:            889,196 ton/yr  top1000=50.7%  n80=4,407
  Without Japan:        668,824 ton/yr  top1000=68.0%  n80=1,991
